In [ ]:
# feature_engineering

from google.colab import drive
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler

print(">>> Mounting Google Drive...")
drive.mount('/content/drive')

spark = SparkSession.builder \
    .appName("HIGGS_FeatureEngineering") \
    .config("spark.executor.memory", "6g") \
    .config("spark.driver.memory", "6g") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

# 1. Load data
raw_input_path = "/content/drive/MyDrive/Bigdata/HIGGS_raw.parquet"
df_raw = spark.read.parquet(raw_input_path)

print("\n>>> STEP 3: Strategic Sampling & Feature Engineering...")

# 2. Upstream Sampling
vraj_sample_df = df_raw.sample(fraction=0.05, seed=42)

# 3. Custom Transformer Pipeline
feature_cols = [f"feature_{i}" for i in range(1, 29)]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")
scaler = StandardScaler(inputCol="raw_features", outputCol="features", withStd=True, withMean=True)

df_assembled = assembler.transform(vraj_sample_df)
df_scaled = scaler.fit(df_assembled).transform(df_assembled)

# 4. Save Engineered Features
features_output_path = "/content/drive/MyDrive/Bigdata/HIGGS_features.parquet"
df_scaled.select("label", "features").write.mode("overwrite").parquet(features_output_path)
print(f"   SUCCESS: Engineered Features saved to {features_output_path} for Model Training.")